In [ ]:
from google.colab import drive
drive.mount('/content/drive')

root_dir = '/content/drive/MyDrive/PCB_defect_detection'

Mounted at /content/drive


In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.8/978.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

import os
import shutil
import random
import xml.etree.ElementTree as ET
import yaml
from pathlib import Path
from collections import Counter

from ultralytics import YOLO
from sklearn.model_selection import KFold

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Explore Dataset

In [ ]:
dataset_dir = os.path.join(root_dir, 'PCB_DATASET')

for root, dirs, files in os.walk(dataset_dir):
   for name in dirs:
      print(os.path.join(root, name))

In [ ]:
def count_files_in_folder(folder_path):
    # Get list of all files in the folder
    files = os.listdir(folder_path)

    # Count the number of files
    num_files = len(files)

    return num_files

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
dataset_dir = '/content/drive/MyDrive/PCB_defect_detection'
subfolders = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']
image_dir = os.path.join(dataset_dir, 'images')
annot_dir = os.path.join(dataset_dir, 'Annotations')

# Function to count files
def count_files_in_folder(folder_path):
    if os.path.exists(folder_path):
        return len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
    return 0

# Print counts for each subfolder
for subfolder in subfolders:
    images_path = os.path.join(image_dir, subfolder)
    annot_path = os.path.join(annot_dir, subfolder)

    print(f'{subfolder:<15} \t'
          f'{count_files_in_folder(images_path)} images \t'
          f'{count_files_in_folder(annot_path)} annotations')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Missing_hole    	115 images 	115 annotations
Mouse_bite      	115 images 	115 annotations
Open_circuit    	116 images 	116 annotations
Short           	116 images 	116 annotations
Spur            	115 images 	115 annotations
Spurious_copper 	116 images 	116 annotations


# Create annotaton dataset

In [ ]:
def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    data = []

    filename = root.find('filename').text
    width = int(root.find('size/width').text)
    height = int(root.find('size/height').text)

    for obj in root.findall('object'):
        name = obj.find('name').text
        xmin = int(obj.find('bndbox/xmin').text)
        ymin = int(obj.find('bndbox/ymin').text)
        xmax = int(obj.find('bndbox/xmax').text)
        ymax = int(obj.find('bndbox/ymax').text)

        data.append({
            'filename': filename,
            'width': width,
            'height': height,
            'class': name,
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax
        })

    return data

In [ ]:
# List to store parsed data from all XML files
all_data = []

# Recursively traverse subdirectories
for root, dirs, files in os.walk(annot_dir):
    for name in files:
        if name.endswith('.xml'):
            xml_path = os.path.join(root, name)
            all_data.extend(parse_xml(xml_path))

In [ ]:
# Create DataFrame from the parsed data
annot_df = pd.DataFrame(all_data)
annot_df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,01_short_07.jpg,3034,1586,short,705,1114,771,1169
1,01_short_07.jpg,3034,1586,short,1645,524,1744,617
2,01_short_07.jpg,3034,1586,short,1599,1227,1688,1312
3,01_short_09.jpg,3034,1586,short,921,1254,997,1318
4,01_short_09.jpg,3034,1586,short,803,1373,865,1432


# Visualization

In [ ]:
def get_subfolder(image_name):
    if 'missing' in image_name.split('_'):
        return 'Missing_hole'
    if 'mouse' in image_name.split('_'):
        return'Mouse_bite'
    if 'open' in image_name.split('_'):
        return 'Open_circuit'
    if 'short' in image_name.split('_'):
        return 'Short'
    if 'spur' in image_name.split('_'):
        return 'Spur'
    if 'spurious' in image_name.split('_'):
        return 'Spurious_copper'

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

def get_subfolder(image_name):
    parts = image_name.split('_')
    if len(parts) >= 2:
        class_name = parts[1].capitalize()  # E.g., 'short' → 'Short'
        return class_name.replace('.jpg', '')
    return 'Unknown'

def visualize_annotations(image_name, images_dir, annot_df, is_subfolder=False):
    # Construct path for image
    if is_subfolder:
        image_path = os.path.join(images_dir, get_subfolder(image_name), image_name)
    else:
        image_path = os.path.join(images_dir, image_name)

    # Read image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Image not found at: {image_path}")
        return None

    # Filter annotations for the current image
    annotations = annot_df[annot_df['filename'] == image_name]
    print(f"Found {len(annotations)} annotations for {image_name}")

    # Draw bounding boxes on the image
    for _, annot in annotations.iterrows():
        xmin, ymin, xmax, ymax = annot['xmin'], annot['ymin'], annot['xmax'], annot['ymax']
        class_label = annot['class']

        # Check if confidence column exists
        confidence = annot.get('confidence')
        if confidence is not None and not pd.isna(confidence):
            class_label += f" ({confidence:.2f})"

        color = (255, 255, 255)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), color, 3)

        # Add background to the text
        text_size = cv2.getTextSize(class_label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 2)[0]
        cv2.rectangle(image, (xmin, ymin - text_size[1] - 5),
                             (xmin + text_size[0], ymin - 1), color, -1)

        # Add text
        cv2.putText(image, class_label, (xmin, ymin - 5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 2)

    # Convert BGR image to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Show image
    plt.figure(figsize=(18, 10))
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.title(f'Annotations - {image_name}')
    plt.show()

    return image


In [ ]:
dataset_dir = '/content/drive/MyDrive/PCB_defect_detection'
images_dir = os.path.join(dataset_dir, 'images')


# Dataset preprocessing

In [ ]:
def resize_images(input_dir, output_dir, target_size=(640, 640)):
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Traverse through the subfolders in the input folder
    for root, _, files in os.walk(input_dir):
        for file in files:
            # Check if the file is an image
            if file.lower().endswith(('.jpg')):
                # Read the image
                image_path = os.path.join(root, file)
                image = cv2.imread(image_path)

                # Resize the image
                resized_image = cv2.resize(image, target_size)

                # Save the resized image to the output folder
                output_path = os.path.join(output_dir, file)
                cv2.imwrite(output_path, resized_image)

resized_img_dir = os.path.join(dataset_dir, 'images_resized')
resize_images(images_dir, resized_img_dir)

In [ ]:
def resize_annotations(annot_df, target_size=(640, 640)):
    all_data = []

    # Iterate through the annotation DataFrame
    for index, row in annot_df.iterrows():

        # Resize the bounding box coordinates
        width_ratio = target_size[0] / row['width']
        height_ratio = target_size[1] / row['height']

        resized_xmin = int(row['xmin'] * width_ratio)
        resized_ymin = int(row['ymin'] * height_ratio)
        resized_xmax = int(row['xmax'] * width_ratio)
        resized_ymax = int(row['ymax'] * height_ratio)

        # Update the all data list with resized annotations
        all_data.append({
            'filename': row['filename'],
            'width': target_size[0],
            'height': target_size[1],
            'class': row['class'],
            'xmin': resized_xmin,
            'ymin': resized_ymin,
            'xmax': resized_xmax,
            'ymax': resized_ymax
        })

    annot_df_resized = pd.DataFrame(all_data)
    return annot_df_resized

annot_df_resized = resize_annotations(annot_df)
annot_df_resized.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,01_short_07.jpg,640,640,short,148,449,162,471
1,01_short_07.jpg,640,640,short,347,211,367,248
2,01_short_07.jpg,640,640,short,337,495,356,529
3,01_short_09.jpg,640,640,short,194,506,210,531
4,01_short_09.jpg,640,640,short,169,554,182,577


# Split dataset

In [ ]:
# Create the output directory
output_dir = os.path.join(dataset_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Convert annotation DataFrame into YOLO labels
# YOLO format: <class_index> <x_center> <y_center> <width> <height>
def convert_to_yolo_labels(annotation_df, classes, target_size=(640, 640)):
    yolo_labels = []

    for _, annot in annotation_df.iterrows():
        filename = annot['filename']
        width, height = annot['width'], annot['height']
        class_name = annot['class']
        xmin, ymin, xmax, ymax = annot['xmin'], annot['ymin'], annot['xmax'], annot['ymax']

        # Convert bounding box coordinates to YOLO format
        x_center = (xmin + xmax) / (2 * width)
        y_center = (ymin + ymax) / (2 * height)
        bbox_width = (xmax - xmin) / width
        bbox_height = (ymax - ymin) / height

        class_index = classes.index(class_name)

        # Append to YOLO labels list
        yolo_labels.append((filename, class_index, x_center, y_center, bbox_width, bbox_height))

    return yolo_labels


classes = ['missing_hole', 'mouse_bite', 'open_circuit',
           'short', 'spur', 'spurious_copper']
yolo_labels = convert_to_yolo_labels(annot_df_resized, classes)


In [ ]:
def split_images_and_labels(images_dir, labels, output_dir, train_split=0.95, val_split=0.05):
    # os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images/train'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images/val'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images/test'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels/train'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels/val'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels/test'), exist_ok=True)

    # Group labels by image filename
    image_labels = {}
    for label in labels:
        filename, class_index, x_center, y_center, bbox_width, bbox_height = label
        if filename not in image_labels:
            image_labels[filename] = []
        image_labels[filename].append(label)

    # Shuffle the image filenames
    image_filenames = list(image_labels.keys())
    random.shuffle(image_filenames)

    # Split the dataset
    num_images = len(image_filenames)
    num_train = int(num_images * train_split)
    num_val = int(num_images * val_split)

    train_filenames = image_filenames[:num_train]
    val_filenames = image_filenames[num_train:num_train + num_val]
    test_filenames = image_filenames[num_train + num_val:]

    # Write train, val, test images and labels
    for dataset, filenames in [('train', train_filenames), ('val', val_filenames), ('test', test_filenames)]:
        for filename in filenames:
            labels = image_labels[filename]
            with open(os.path.join(output_dir, f'labels/{dataset}/{os.path.splitext(filename)[0]}.txt'), 'a') as label_file:
                for label in labels:
                    _, class_index, x_center, y_center, bbox_width, bbox_height = label
                    label_file.write(f"{class_index} {x_center} {y_center} {bbox_width} {bbox_height}\n")
            # Copy images to corresponding folders
            shutil.copy(os.path.join(images_dir, filename), os.path.join(output_dir, f'images/{dataset}/{filename}'))

split_images_and_labels(resized_img_dir, yolo_labels, output_dir)

# K-Fold Cross Validation

In [ ]:
dataset_path = Path(output_dir)
labels = sorted(dataset_path.rglob("*labels/train/*.txt")) # all data in 'labels'

In [ ]:
cls_idx = list(range(len(classes)))
print(list(zip(classes, cls_idx)))

[('missing_hole', 0), ('mouse_bite', 1), ('open_circuit', 2), ('short', 3), ('spur', 4), ('spurious_copper', 5)]


In [ ]:
indx = [l.stem for l in labels] # uses base filename as ID (no extension)
labels_df = pd.DataFrame([], columns=cls_idx, index=indx)

In [ ]:
for label in labels:
    lbl_counter = Counter()

    with open(label,'r') as lf:
        lines = lf.readlines()

    for l in lines:
        # classes for YOLO label uses integer at first position of each line
        lbl_counter[int(l.split(' ')[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0) # replace `nan` values with `0.0`
labels_df.head()

<ipython-input-23-635b1554578e>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels_df = labels_df.fillna(0.0) # replace `nan` values with `0.0`


,0,1,2,3,4,5
01_missing_hole_01,6.0,0.0,0.0,0.0,0.0,0.0
01_missing_hole_02,6.0,0.0,0.0,0.0,0.0,0.0
01_missing_hole_03,6.0,0.0,0.0,0.0,0.0,0.0
01_missing_hole_04,6.0,0.0,0.0,0.0,0.0,0.0
01_missing_hole_05,8.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
ksplit = 3
kf = KFold(n_splits=ksplit, shuffle=True, random_state=20)   # setting random_state for repeatable results

kfolds = list(kf.split(labels_df))

In [ ]:
folds = [f'split_{n}' for n in range(1, ksplit + 1)]
folds_df = pd.DataFrame(index=indx, columns=folds)

for idx, (train, val) in enumerate(kfolds, start=1):
    folds_df[f'split_{idx}'].loc[labels_df.iloc[train].index] = 'train'
    folds_df[f'split_{idx}'].loc[labels_df.iloc[val].index] = 'val'

<ipython-input-25-68702af6ce23>:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  folds_df[f'split_{idx}'].loc[labels_df.iloc[train].index] = 'train'
<ipython-input-25-68702af6ce23>:6: FutureWarning: ChainedAssignmentError: behaviour will cha

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1E-7)
    fold_lbl_distrb.loc[f'split_{n}'] = ratio

fold_lbl_distrb

,0,1,2,3,4,5
split_1,0.583193,0.517799,0.44113,0.434916,0.462158,0.572581
split_2,0.509615,0.589831,0.522388,0.533552,0.565517,0.379066
split_3,0.416541,0.404192,0.540268,0.536066,0.476423,0.565008


In [ ]:
# Initialize a list to store image file paths
images = sorted(dataset_path.rglob("*images/train/*.jpg"))

# Create the necessary directories and dataset YAML files (unchanged)
save_path = Path(dataset_path / f'{ksplit}fold_crossval')
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / 'train' / 'images').mkdir(parents=True, exist_ok=True)
    (split_dir / 'train' / 'labels').mkdir(parents=True, exist_ok=True)
    (split_dir / 'val' / 'images').mkdir(parents=True, exist_ok=True)
    (split_dir / 'val' / 'labels').mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f'{split}_dataset.yaml'
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, 'w') as ds_y:
        yaml.safe_dump({
            'path': split_dir.as_posix(),
            'train': 'train',
            'val': 'val',
            'names': classes
        }, ds_y)

In [ ]:
for image, label in zip(images, labels):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / split / k_split / 'images'
        lbl_to_path = save_path / split / k_split / 'labels'

        # Copy image and label files to new directory
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

In [ ]:
folds_df.to_csv(save_path / "kfold_datasplit.csv")
fold_lbl_distrb.to_csv(save_path / "kfold_label_distribution.csv")

# Model Training

In [ ]:
model = YOLO('yolov8s.pt')
results = {}

# Additional arguments here
batch = 16
project = 'pcb'
epochs = 100
imgsz=640
save_period=1
verbose=True
# box=6.0 # Weight of the box loss component in the loss function
# cls = 2.0 # Weight of the classification loss in the total loss function
mixup = 0.3 # Blends two images and their labels, creating a composite image

for k in range(ksplit):
    dataset_yaml = ds_yamls[k]
    model.train(data=dataset_yaml,
                epochs=epochs,
                batch=batch,
                lr0=0.001,
                lrf=0.0001,
                imgsz=imgsz,
                save_period=save_period,
                verbose=verbose,
                project=project,
                mixup=mixup)
    results[k] = model.metrics  # save output metrics for further analysis

Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/split_1_dataset.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=1, cache=False, device=None, workers=8, project=pcb, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=F

train: Scanning /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/labels.cache... 597 images, 0 backgrounds, 0 corrupt: 100%|██████████| 597/597 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_01.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_03.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_05.jpg: 4 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_06.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_08.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/train/images/01_missing_hole_12.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/ou

val: Fast image access ✅ (ping: 3.0±3.7 ms, read: 49.3±34.6 MB/s, size: 160.1 KB)


val: Scanning /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/labels.cache... 372 images, 0 backgrounds, 0 corrupt: 100%|██████████| 372/372 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_02.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_04.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_09.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_11.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_14.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_1/val/images/01_missing_hole_15.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_

... (305 outputs left out)

val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_missing_hole_01.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_missing_hole_03.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_missing_hole_08.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_missing_hole_20.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_mouse_bite_03.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/val/images/01_mouse_bite_06.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/3fold_crossval/split_2/va

Plotting labels to pcb/train42/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to pcb/train42
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      4.19G      1.578     0.9198     0.9183         72        640:  61%|██████    | 23/38 [00:11<00:07,  2.09it/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-46-9e531cf955e4>", line 17, in <cell line: 0>
    model.train(data=dataset_yaml,
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/model.py", line 791, in train
    self.trainer.train()
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/trainer.py", line 211, in train
    self._do_train(world_size)
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/trainer.py", line 385, in _do_train
    loss, self.loss_items = self.model(batch)
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn

# Model

In [ ]:
model = YOLO('yolov8s.pt')

# Additional arguments here
batch = 16
project = 'pcb'
epochs = 100
imgsz=640
save_period=1
verbose=True
# box=6.0 # Weight of the box loss component in the loss function
# cls = 2.0 # Weight of the classification loss in the total loss function
mixup = 0.3 # Blends two images and their labels, creating a composite image

all_data_yaml = f"""
path: {output_dir}
train: images/train
val: images/val

names:
    0: missing_hole
    1: mouse_bite
    2: open_circuit
    3: short
    4: spur
    5: spurious_copper
"""

data_path = os.path.join(root_dir, 'data.yaml')

with open(data_path, 'w') as f:
    f.write(all_data_yaml)

result = model.train(data=data_path,
                     epochs=epochs,
                     batch=batch,
                     lr0=0.001,
                     lrf=0.0001,
                     imgsz=imgsz,
                     save_period=save_period,
                     verbose=verbose,
                     project=project,
                     mixup=mixup)

Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/drive/MyDrive/PCB_defect_detection/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=1, cache=False, device=None, workers=8, project=pcb, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, s

train: Scanning /content/drive/MyDrive/PCB_defect_detection/output/labels/train... 691 images, 0 backgrounds, 0 corrupt: 100%|██████████| 691/691 [00:07<00:00, 97.50it/s] 

train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_01.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_02.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_03.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_04.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_05.jpg: 4 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_06.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/train/01_missing_hole_08.jpg: 3 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/

train: New cache created: /content/drive/MyDrive/PCB_defect_detection/output/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 1.0±0.5 ms, read: 8.3±11.0 MB/s, size: 165.3 KB)


val: Scanning /content/drive/MyDrive/PCB_defect_detection/output/labels/val... 66 images, 0 backgrounds, 0 corrupt: 100%|██████████| 66/66 [00:22<00:00,  2.95it/s]

val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/val/04_mouse_bite_19.jpg: 3 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/PCB_defect_detection/output/images/val/06_mouse_bite_02.jpg: 6 duplicate labels removed
val: New cache created: /content/drive/MyDrive/PCB_defect_detection/output/labels/val.cache


... (286 outputs left out)

                   all         66        271      0.987      0.941      0.979      0.604



100 epochs completed in 0.609 hours.
Optimizer stripped from pcb/train3/weights/last.pt, 22.5MB
Optimizer stripped from pcb/train3/weights/best.pt, 22.5MB

Validating pcb/train3/weights/best.pt...
Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.40it/s]


                   all         66        271      0.986      0.945      0.979      0.613
          missing_hole         12         52      0.992          1      0.995      0.671
            mouse_bite         10         37      0.969      0.846      0.938      0.553
          open_circuit         10         46      0.997          1      0.995       0.55
                 short         11         42      0.991          1      0.995      0.672
                  spur         17         68      0.964      0.882      0.957      0.575
       spurious_copper          6         26          1      0.938      0.995      0.656
Speed: 0.2ms preprocess, 5.8ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to pcb/train3


In [ ]:
results_dir = '/content/pcb/train4'
dest_results_dir = os.path.join(root_dir, 'results')

shutil.copytree(results_dir, dest_results_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/content/pcb/train4'

In [ ]:
results_df = pd.read_csv(os.path.join(dest_results_dir, 'results.csv'))
results_df.columns = results_df.columns.str.strip()
results_df = results_df.apply(pd.to_numeric, errors='coerce').dropna()
results_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PCB_defect_detection/results/results.csv'

In [ ]:
epochs = results_df['epoch']
train_box_loss = results_df['train/box_loss']
val_box_loss = results_df['val/box_loss']
train_cls_loss = results_df['train/cls_loss']
val_cls_loss = results_df['val/cls_loss']
train_dfl_loss = results_df['train/dfl_loss']
val_dfl_loss = results_df['val/dfl_loss']

# Create a figure with three subplots
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

# Plot box loss
axs[0].plot(epochs, train_box_loss, label='Train Box Loss', color='blue')
axs[0].plot(epochs, val_box_loss, label='Validation Box Loss', color='orange')
axs[0].set_title('Box Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[0].grid(True)

# Plot cls loss
axs[1].plot(epochs, train_cls_loss, label='Train Cls Loss', color='blue')
axs[1].plot(epochs, val_cls_loss, label='Validation Cls Loss', color='orange')
axs[1].set_title('Class Loss')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].legend()
axs[1].grid(True)

# Plot dfl loss
axs[2].plot(epochs, train_dfl_loss, label='Train Dfl Loss', color='blue')
axs[2].plot(epochs, val_dfl_loss, label='Validation Dfl Loss', color='orange')
axs[2].set_title('Distribution Focal Loss')
axs[2].set_xlabel('Epoch')
axs[2].set_ylabel('Loss')
axs[2].legend()
axs[2].grid(True)

plt.tight_layout()
plt.show()

# Predict on test data

In [ ]:
best_model_path = os.path.join(dest_results_dir, 'weights/best.pt')
model = YOLO(best_model_path)

test_data_dir = os.path.join(output_dir, 'images/val')
metrics = model(source=test_data_dir, imgsz=640, conf=0.25, save=True, save_txt=True, save_conf=True)

In [ ]:
predict_dir = '/content/runs/detect/predict'
dest_predict_dir = os.path.join(root_dir, 'results/predict')

shutil.copytree(predict_dir, dest_predict_dir)

In [ ]:
def yolo_to_original_annot(image_name, yolo_labels, annot_df, classes):
    original_annot = []

    for yolo_label in yolo_labels:
        # Extract original width and height from annotation DataFrame
        original_size = annot_df.loc[annot_df['filename'] == image_name, ['width', 'height']].iloc[0]
        original_width, original_height = original_size['width'], original_size['height']

        # Extract YOLO label components
        class_index, x_center, y_center, bbox_width, bbox_height, confidence = yolo_label

        # Scale bounding box coordinates and dimensions to original size
        original_x_center = x_center * original_width
        original_y_center = y_center * original_height
        original_bbox_width = bbox_width * original_width
        original_bbox_height = bbox_height * original_height

        # Calculate original bounding box coordinates
        original_x_min = original_x_center - original_bbox_width / 2
        original_y_min = original_y_center - original_bbox_height / 2
        original_x_max = original_x_center + original_bbox_width / 2
        original_y_max = original_y_center + original_bbox_height / 2

        # Append original annotation to list
        original_annot.append({
            'filename': image_name,
            'width': int(original_width),
            'height': int(original_height),
            'class': classes[int(class_index)],
            'xmin': int(original_x_min),
            'ymin': int(original_y_min),
            'xmax': int(original_x_max),
            'ymax': int(original_y_max),
            'confidence': confidence
        })

    return pd.DataFrame(original_annot)

In [ ]:
def read_yolo_labels_from_file(file_path):
    labels = []
    with open(file_path, 'r') as file:
        for line in file:
            values = line.strip().split()
            values = [float(value) for value in values]
            labels.append(values)
    return labels

file_path = os.path.join(dest_predict_dir, 'labels/12_spurious_copper_10.txt')
yolo_labels = read_yolo_labels_from_file(file_path)
yolo_labels

In [ ]:
pred_annot_df = yolo_to_original_annot('12_spurious_copper_10.jpg', yolo_labels, annot_df, classes)
pred_annot_df.head()

In [ ]:
visualize_annotations('12_spurious_copper_10.jpg', images_dir, pred_annot_df, is_subfolder=True);

In [ ]:
visualize_annotations('12_spurious_copper_10.jpg', images_dir, annot_df, is_subfolder=True);

Image not found at: /content/drive/MyDrive/PCB_defect_detection/images/Spurious/12_spurious_copper_10.jpg


In [ ]:
model.export()

Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.00GHz)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from 'pcb/train3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (21.5 MB)

TorchScript: starting export with torch 2.6.0+cu124...
TorchScript: export success ✅ 3.3s, saved as 'pcb/train3/weights/best.torchscript' (42.9 MB)

Export complete (5.9s)
Results saved to /content/pcb/train3/weights
Predict:         yolo predict task=detect model=pcb/train3/weights/best.torchscript imgsz=640  
Validate:        yolo val task=detect model=pcb/train3/weights/best.torchscript imgsz=640 data=/content/drive/MyDrive/PCB_defect_detection/data.yaml  
Visualize:       https://netron.app


'pcb/train3/weights/best.torchscript'

In [ ]:
custom_img_dir = os.path.join(root_dir, 'custom_images')
resized_custom_img_dir = os.path.join(custom_img_dir, 'resized')
resize_images(custom_img_dir, resized_custom_img_dir)

In [ ]:
model = YOLO(best_model_path)

image_path = os.path.join(resized_custom_img_dir, '01.jpg')
result_custom = model(image_path, imgsz=640, conf=0.25, save=True, save_txt=True, save_conf=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PCB_defect_detection/results/weights/best.pt'

In [ ]:
predict_dir = '/content/runs/detect/predict2'
dest_custom_predict_dir = os.path.join(custom_img_dir, 'results/predict')

shutil.copytree(predict_dir, dest_custom_predict_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/predict2'